# ROI Pattern ISC — Left-Wing Subjects (Auditory & Visual)

**Goal**: Measure inter-subject pattern correlation (ISPC) within visual and auditory ROIs
for each of the four political-content conditions, using left-wing subjects only.

**Analysis logic** (Chen et al., 2017, *Nature Neuroscience*, doi:10.1038/nn.4450):

1. For each subject and condition, load the pre-extracted NPZ `(n_posts, n_voxels)`
2. Average across posts → one spatial pattern `(n_voxels,)` per subject
3. Leave-one-out ISC: for each subject i, compute Pearson r between their pattern
   and the mean of all others — **correlation is across voxels** (spatial similarity),
   not across posts
4. Average r across subjects → one ISC value per condition × ROI
5. Permutation test (subject-label shuffle) + one-sample t-test vs. zero

**Key difference from `parcellated_leftwing_ispc.ipynb`**:  
That notebook correlates parcel activation *across posts* (timecourse covariance).  
This notebook correlates voxel activation *across space* (pattern similarity within an ROI).

**ROIs**:
- Auditory: `pieman_a1_2mm.nii` (2600 voxels)
- Visual: `primary visual_association-test_z_FDR_0.01.nii.gz` (1831 voxels)

**Conditions**: AntiLeft · AntiRight · ProLeft · ProRight

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

# Make sure the package is importable when running from the examples/ folder
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from yy_fmri_kit.event_isc.roi_similarity import (
    run_roi_isc,
    results_to_dataframe,
)

## 1. Configuration

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
ROOT           = Path("/path/to/your/project/root")  # CHANGE THIS
BEHAVIORAL_CSV = ROOT / "behavioral_analyses/data/250226/merged_behavioral_bids.csv"

NPZ_DIRS = {
    "auditory": ROOT / "data/derivatives/postbypost/roi",
    "visual":   ROOT / "data/derivatives/postbypost/visual",
}

OUTPUT_DIR = ROOT / "data/derivatives/roi_ispc/leftwing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── analysis parameters ────────────────────────────────────────────────────
RUN_TYPES = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"]
N_PERMS   = 1000
SEED      = 42

## 2. Load Left-Wing Subjects

In [ ]:
beh_df   = pd.read_csv(BEHAVIORAL_CSV)
beh_df   = beh_df.drop_duplicates(subset='bids_id', keep='first')
subjects = sorted(beh_df[beh_df['political_group'] == 'left']['bids_id'].tolist())

print(f"Left-wing subjects ({len(subjects)}):")
print(subjects)

## 3. Run Pattern ISC — All Conditions × Both ROIs

In [ ]:
results_by_roi = {}

for roi_name, npz_dir in NPZ_DIRS.items():
    results_by_roi[roi_name] = run_roi_isc(
        npz_dir   = npz_dir,
        subjects  = subjects,
        run_types = RUN_TYPES,
        roi_name  = roi_name,
        n_perms   = N_PERMS,
        seed      = SEED,
    )

## 4. Results Table

In [ ]:
df = results_to_dataframe(results_by_roi)

# Flag significance (both tests)
df['sig_perm']  = df['p_perm']  < 0.05
df['sig_ttest'] = df['p_ttest'] < 0.05

print(df.to_string(index=False))

## 5. Per-Subject ISC Values

In [ ]:
# Inspect per-subject ISC for each condition × ROI
for roi_name, results in results_by_roi.items():
    print(f"\n{'='*50}")
    print(f"ROI: {roi_name.upper()}")
    print(f"{'='*50}")
    for condition, r in results.items():
        subj_vals = r['isc_subj']
        subs      = r['subjects']
        print(f"\n  {condition}  (mean r = {r['isc_mean']:.4f})")
        for sub, val in zip(subs, subj_vals):
            print(f"    {sub}: r = {val:.4f}")

## 6. Visualisation — ISC per Condition × ROI

In [ ]:
roi_names  = list(results_by_roi.keys())   # ['auditory', 'visual']
conditions = RUN_TYPES
n_rois     = len(roi_names)
n_conds    = len(conditions)

fig, axes = plt.subplots(1, n_rois, figsize=(5 * n_rois, 5), sharey=True)
if n_rois == 1:
    axes = [axes]

colors = ['#E07B54', '#5B8DB8', '#6BAE75', '#A97FC4']

for ax, roi_name in zip(axes, roi_names):
    results = results_by_roi[roi_name]
    means   = [results[c]['isc_mean']  for c in conditions if c in results]
    sems    = [
        results[c]['isc_subj'].std() / np.sqrt(results[c]['n_subjects'])
        for c in conditions if c in results
    ]
    conds_present = [c for c in conditions if c in results]

    bars = ax.bar(conds_present, means, yerr=sems, capsize=4,
                  color=colors[:len(conds_present)], edgecolor='k', linewidth=0.7)

    # Mark significance with * (permutation p < 0.05)
    for i, cond in enumerate(conds_present):
        p = results[cond]['p_perm']
        if p < 0.001:
            label = '***'
        elif p < 0.01:
            label = '**'
        elif p < 0.05:
            label = '*'
        else:
            label = 'ns'
        y_pos = means[i] + sems[i] + 0.005
        ax.text(i, y_pos, label, ha='center', va='bottom', fontsize=10)

    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_title(f'{roi_name.capitalize()} ROI', fontsize=13, fontweight='bold')
    ax.set_ylabel('Mean ISC (Pearson r)' if ax == axes[0] else '')
    ax.set_xlabel('Condition')
    ax.set_xticklabels(conds_present, rotation=20, ha='right')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    'Voxel-Pattern ISC — Left-Wing Subjects\n'
    '(* p<0.05, ** p<0.01, *** p<0.001, permutation test)',
    fontsize=12, y=1.02
)
plt.tight_layout()

fig_path = OUTPUT_DIR / 'roi_ispc_leftwing_barplot.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## 7. Null Distribution Check

In [ ]:
# Plot observed ISC vs. null distribution for each condition × ROI
fig, axes = plt.subplots(n_rois, n_conds, figsize=(4 * n_conds, 3.5 * n_rois),
                         sharey='row')

for row, roi_name in enumerate(roi_names):
    results = results_by_roi[roi_name]
    for col, cond in enumerate(conditions):
        ax = axes[row, col] if n_rois > 1 else axes[col]
        if cond not in results:
            ax.set_visible(False)
            continue
        r         = results[cond]
        null_dist = r['null_dist']
        observed  = r['isc_mean']

        ax.hist(null_dist, bins=40, color='lightgrey', edgecolor='k', linewidth=0.4)
        ax.axvline(observed, color='crimson', linewidth=2,
                   label=f'obs r={observed:.3f}\np={r["p_perm"]:.3f}')
        ax.legend(fontsize=8, loc='upper right')
        ax.set_title(f'{roi_name} / {cond}', fontsize=9)
        ax.set_xlabel('ISC (null)' if row == n_rois - 1 else '')
        ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Observed ISC vs. Null Distribution (subject-label permutation)',
             fontsize=12, y=1.01)
plt.tight_layout()

null_fig_path = OUTPUT_DIR / 'roi_ispc_null_distributions.png'
fig.savefig(null_fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {null_fig_path}')
plt.show()

## 8. Save Results

In [ ]:
# Summary table (one row per condition × ROI)
out_csv = OUTPUT_DIR / 'roi_ispc_leftwing_summary.csv'
df.to_csv(out_csv, index=False)
print(f'Summary saved → {out_csv}')

# Per-subject ISC values (one file per ROI)
for roi_name, results in results_by_roi.items():
    rows = []
    for condition, r in results.items():
        for sub, val in zip(r['subjects'], r['isc_subj']):
            rows.append({'roi': roi_name, 'condition': condition,
                         'subject': sub, 'isc_r': round(float(val), 6)})
    subj_df = pd.DataFrame(rows)
    subj_csv = OUTPUT_DIR / f'roi_ispc_{roi_name}_persubject.csv'
    subj_df.to_csv(subj_csv, index=False)
    print(f'Per-subject saved → {subj_csv}')

df

## 9. Build ISC NIfTI Maps

For each condition, paint the mean ISC value from each ROI into its mask voxels,
producing a single NIfTI volume. This is used by both yabplot and nilearn.

In [ ]:
import nibabel as nib
import numpy as np

MASK_PATHS = {
    "auditory": ROOT / "data/brain_masks/pieman_a1_2mm.nii",
    "visual":   ROOT / "data/brain_masks/primary visual_association-test_z_FDR_0.01.nii.gz",
}
REF_IMG = ROOT / "data/brain_masks/MNI152_T1_2mm_brain.nii"

def isc_to_nifti(results_by_roi, mask_paths, condition, ref_img_path):
    """
    Paint mean ISC values into ROI voxels of a blank MNI volume.
    Each ROI gets a uniform fill equal to its mean ISC.
    """
    ref_img  = nib.load(str(ref_img_path))
    out_data = np.zeros(ref_img.shape[:3], dtype=np.float32)
    for roi_name, mask_path in mask_paths.items():
        if roi_name not in results_by_roi:
            continue
        if condition not in results_by_roi[roi_name]:
            continue
        isc_val  = float(results_by_roi[roi_name][condition]["isc_mean"])
        mask_img = nib.load(str(mask_path))
        # resample mask to reference space if needed
        from nilearn.image import resample_to_img
        mask_res = resample_to_img(mask_img, ref_img, interpolation="nearest")
        roi_bool = mask_res.get_fdata(dtype=np.float32) > 0
        out_data[roi_bool] = isc_val
    return nib.Nifti1Image(out_data, ref_img.affine, ref_img.header)

# Build one NIfTI per condition and save
NIFTI_DIR = OUTPUT_DIR / "nifti_maps"
NIFTI_DIR.mkdir(exist_ok=True)

nifti_paths = {}
for cond in RUN_TYPES:
    nii = isc_to_nifti(results_by_roi, MASK_PATHS, cond, REF_IMG)
    out_path = NIFTI_DIR / f"roi_isc_{cond}.nii.gz"
    nib.save(nii, str(out_path))
    nifti_paths[cond] = out_path
    print(f"{cond}: saved → {out_path.name}  (non-zero voxels: {(nii.get_fdata() != 0).sum()})")

## 9b. Yabplot — Static Surface Brain Maps

Projects each condition's ISC NIfTI onto the fsLR-32k midthickness surface using `yabplot.project_vol2surf`. Colour scale is shared across all conditions so maps are directly comparable.

In [ ]:
import yabplot as yab
import yabplot.data as ydata

_lh_surf, _rh_surf = ydata.get_surface_paths("midthickness", "bmesh")

ALL_VIEWS = [
    "left_lateral", "left_medial", "right_lateral", "right_medial",
    "superior", "inferior", "anterior", "posterior",
]

# Shared colour scale: symmetric around 0, driven by max |ISC| across all conditions
all_isc = [
    results_by_roi[roi][cond]["isc_mean"]
    for roi in results_by_roi
    for cond in results_by_roi[roi]
]
VMAX     = max(abs(v) for v in all_isc)
VMAX     = max(VMAX, 0.01)   # floor so colour scale is never degenerate
VMINMAX  = [-VMAX, VMAX]
print(f"Shared colour scale: [{-VMAX:.4f}, {VMAX:.4f}]")

STATIC_DIR = OUTPUT_DIR / "brain_maps_static"
STATIC_DIR.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation="nearest")
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    fig = yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = "RdBu_r",
        vminmax      = VMINMAX,
        figsize      = (1600, 800),
        display_type = "static",
        export_path  = str(STATIC_DIR / f"brain_isc_{cond}.png"),
    )
    print(f"{cond}: saved → brain_isc_{cond}.png")

## 10. Nilearn — Interactive HTML Brain Maps

Generates an interactive HTML viewer per condition using `nilearn.plotting.view_img`. Each map shows both ROIs painted with their mean ISC value.

In [ ]:
from nilearn import plotting
from IPython.display import IFrame, display

HTML_DIR = OUTPUT_DIR / "brain_maps_html"
HTML_DIR.mkdir(exist_ok=True)

EPS = 1e-6   # small threshold so zero-value voxels are transparent

for cond, nii_path in nifti_paths.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = "MNI152",
        cmap           = "RdBu_r",
        threshold      = EPS,
        vmin           = -VMAX,
        vmax           =  VMAX,
        title          = f"ROI Pattern ISC — {cond} (left-wing)",
        symmetric_cmap = False,
    )
    out_path = HTML_DIR / f"roi_isc_{cond}.html"
    html_view.save_as_html(str(out_path))
    print(f"{cond}: saved → {out_path.name}")

# Display first condition inline
first_html = HTML_DIR / f"roi_isc_{RUN_TYPES[0]}.html"
display(IFrame(str(first_html), width="100%", height=500))

---

## Approach B: Post-wise ISC (Chen et al. 2017)

Approach A (sections above) averages each subject's post patterns into a single
condition-level mean before computing inter-subject correlations.

**Approach B** computes a separate LOO ISC for every post, then averages the
per-post ISC values — matching the structure of Chen et al. (2017).

See the `parcel_ispc_leftwing.ipynb` notebook for a full description of the
methodological trade-offs between the two approaches.


In [ ]:
from yy_fmri_kit.event_isc.roi_similarity import run_roi_isc_postwise

results_b_by_roi = {}
for roi_name, npz_dir in NPZ_DIRS.items():
    results_b_by_roi[roi_name] = run_roi_isc_postwise(
        npz_dir   = npz_dir,
        subjects  = subjects,
        run_types = RUN_TYPES,
        roi_name  = roi_name,
    )


## B.1 Results Table

In [ ]:
rows_b = []
for roi_name, results in results_b_by_roi.items():
    for cond, r in results.items():
        rows_b.append({
            'roi'       : roi_name,
            'condition' : cond,
            'isc_mean'  : round(r['isc_mean'], 6),
            'n_subjects': r['n_subjects'],
            'n_posts'   : r['n_posts'],
            'n_voxels'  : r['n_voxels'],
        })
df_b = pd.DataFrame(rows_b)
print("Approach B — post-wise ISC summary:")
print(df_b.to_string(index=False))
df_b


## B.2 Bar Chart — ISC per Condition × ROI

In [ ]:
n_rois = len(results_b_by_roi)
fig, axes = plt.subplots(1, n_rois, figsize=(6 * n_rois, 5), sharey=True)
if n_rois == 1:
    axes = [axes]

colors = ['#E07B54', '#5B8DB8', '#6AA96B', '#9B6BB5']

for ax, (roi_name, results) in zip(axes, results_b_by_roi.items()):
    conds = [c for c in RUN_TYPES if c in results]
    means = [results[c]['isc_mean'] for c in conds]
    sems  = [results[c]['isc_subj'].std() / np.sqrt(results[c]['n_subjects'])
             for c in conds]
    ax.bar(conds, means, yerr=sems, capsize=4,
           color=colors[:len(conds)], alpha=0.8,
           edgecolor='black', linewidth=0.7)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'Approach B — {roi_name.capitalize()} ROI', fontsize=12)
    ax.set_xlabel('Condition', fontsize=10)
    ax.set_ylabel('Mean ISC (r)', fontsize=10)
    ax.tick_params(axis='x', rotation=20)

fig.suptitle('Approach B: Post-wise Pattern ISC per Condition × ROI',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roi_isc_B_barplot.png', dpi=150, bbox_inches='tight')
plt.show()


## B.3 Build ISC NIfTI Maps

In [ ]:
# isc_to_nifti is defined in cell 19 above — reused here for Approach B
NIFTI_DIR_B = OUTPUT_DIR / "nifti_maps_B"
NIFTI_DIR_B.mkdir(exist_ok=True)

nifti_paths_b = {}
for cond in RUN_TYPES:
    nii = isc_to_nifti(results_b_by_roi, MASK_PATHS, cond, REF_IMG)
    out_path = NIFTI_DIR_B / f"roi_isc_B_{cond}.nii.gz"
    nib.save(nii, str(out_path))
    nifti_paths_b[cond] = out_path
    print(f"{cond}: saved → {out_path.name}  (non-zero voxels: {(nii.get_fdata() != 0).sum()})")


## B.4 Yabplot — Static Surface Brain Maps

In [ ]:
# Shared colour scale for Approach B
all_isc_b = [
    results_b_by_roi[roi][cond]["isc_mean"]
    for roi in results_b_by_roi
    for cond in results_b_by_roi[roi]
]
VMAX_B    = max(abs(v) for v in all_isc_b)
VMAX_B    = max(VMAX_B, 0.01)
VMINMAX_B = [-VMAX_B, VMAX_B]
print(f"Approach B colour scale: [{-VMAX_B:.4f}, {VMAX_B:.4f}]")

STATIC_DIR_B = OUTPUT_DIR / "brain_maps_static_B"
STATIC_DIR_B.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths_b.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation="nearest")
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    fig = yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = "RdBu_r",
        vminmax      = VMINMAX_B,
        figsize      = (1600, 800),
        display_type = "static",
        export_path  = str(STATIC_DIR_B / f"brain_isc_B_{cond}.png"),
    )
    print(f"{cond}: saved → brain_isc_B_{cond}.png")


## B.5 Nilearn — Interactive HTML Brain Maps

In [ ]:
HTML_DIR_B = OUTPUT_DIR / "brain_maps_html_B"
HTML_DIR_B.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths_b.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = "MNI152",
        cmap           = "RdBu_r",
        threshold      = EPS,
        vmin           = -VMAX_B,
        vmax           =  VMAX_B,
        title          = f"ROI Pattern ISC — Approach B — {cond} (left-wing)",
        symmetric_cmap = False,
    )
    out_path = HTML_DIR_B / f"roi_isc_B_{cond}.html"
    html_view.save_as_html(str(out_path))
    print(f"{cond}: saved → {out_path.name}")

# Display first condition inline
display(IFrame(str(HTML_DIR_B / f"roi_isc_B_{RUN_TYPES[0]}.html"), width="100%", height=500))


---

## Approach A vs B: Comparison

With only 4 conditions per ROI, each scatter has 4 labelled points.  Both
methods should rank conditions similarly if the signal is robust; a systematic
upward shift of Approach A is expected (averaging before correlating reduces
noise, inflating r slightly).


In [ ]:
n_rois = len(results_b_by_roi)
fig, axes = plt.subplots(1, n_rois, figsize=(5 * n_rois, 5))
if n_rois == 1:
    axes = [axes]

roi_colors = {'auditory': '#E07B54', 'visual': '#5B8DB8'}

for ax, roi_name in zip(axes, results_b_by_roi.keys()):
    color = roi_colors.get(roi_name, '#888888')
    roi_res_a = results_by_roi.get(roi_name, {})
    roi_res_b = results_b_by_roi.get(roi_name, {})
    conds = [c for c in RUN_TYPES if c in roi_res_a and c in roi_res_b]

    a_vals = [roi_res_a[c]['isc_mean'] for c in conds]
    b_vals = [roi_res_b[c]['isc_mean'] for c in conds]

    ax.scatter(a_vals, b_vals, color=color, s=90, zorder=3, edgecolors='black', linewidths=0.5)
    for x, y, label in zip(a_vals, b_vals, conds):
        ax.annotate(label, (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=9)

    lim = max((abs(v) for v in a_vals + b_vals), default=0.1) * 1.4
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, alpha=0.5, label='identity')
    ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Approach A (mean-pattern ISC)', fontsize=10)
    ax.set_ylabel('Approach B (post-wise ISC)', fontsize=10)
    ax.set_title(f'{roi_name.capitalize()} ROI', fontsize=12)
    ax.set_aspect('equal')

fig.suptitle('Approach A vs B: Mean ISC per Condition × ROI', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roi_isc_AB_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## Save Approach B Results

In [ ]:
df_b.to_csv(OUTPUT_DIR / 'roi_isc_B_summary.csv', index=False)
print("Saved roi_isc_B_summary.csv")

for roi_name, results in results_b_by_roi.items():
    for cond, r in results.items():
        subj_df = pd.DataFrame({'subject': r['subjects'], 'isc': r['isc_subj']})
        out = OUTPUT_DIR / f'roi_isc_B_{roi_name}_{cond}_persubject.csv'
        subj_df.to_csv(out, index=False)
    print(f"  Saved per-subject CSVs for {roi_name}")
